 # Energy Analysis



 The scHopfield energy functional decomposes into three biologically interpretable

 components:



 $$E = E_{\text{interaction}} + E_{\text{degradation}} + E_{\text{bias}}$$



 - **E_interaction**: Energy stored in gene–gene regulatory interactions

 - **E_degradation**: Energy stored in mRNA decay terms

 - **E_bias**: Energy stored in external input / basal expression bias



 Lower total energy ≈ more stable attractor state.  This notebook shows how to

 compute, visualise, and interpret these energies.

 ## Setup

In [ ]:
import itertools

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scHopfield as sch

# Assumes the model was saved in notebook 01
DATA_PATH  = '/path/to/data/'
DATASET_FILE = 'hematopoiesis.h5ad'
MODEL_FILE = 'model.h5sch'

CLUSTER_KEY = 'cell_type'
SPLICED_KEY  = 'M_t'

CELL_TYPE_ORDER = ['Meg', 'Ery', 'MEP-like', 'HSC', 'GMP-like', 'Mon', 'Bas', 'Neu']

adata = sc.read_h5ad(DATA_PATH + DATASET_FILE)
sch.tl.load_model(adata, MODEL_FILE)
print(adata)


 ## 2.1 Compute & Decompose Energies

In [ ]:
sch.tl.compute_energies(adata, cluster_key=CLUSTER_KEY)

# Inspect stored energy columns
energy_cols = ['energy_total', 'energy_interaction', 'energy_degradation', 'energy_bias']
print(adata.obs[energy_cols].describe())


In [ ]:
# Summary statistics per cell type (including coefficient of variation)
summary = adata.obs[[CLUSTER_KEY] + energy_cols].groupby(CLUSTER_KEY).describe()
for energy in energy_cols:
    summary[(energy, 'cv')] = summary[(energy, 'std')] / summary[(energy, 'mean')]

print("\nTotal energy statistics:")
print(summary['energy_total'].round(3))


 ## 2.2 Energy Boxplots by Cell Type

In [ ]:
# Retrieve cell-type colours from the dataset (or define your own dict)
colors = {ct: f'C{i}' for i, ct in enumerate(CELL_TYPE_ORDER)}

# All four energy components
sch.pl.plot_energy_boxplots(
    adata,
    cluster_key=CLUSTER_KEY,
    order=CELL_TYPE_ORDER,
    colors=[colors[k] for k in CELL_TYPE_ORDER]
)
plt.show()


In [ ]:
# Total energy only
sch.pl.plot_energy_boxplots(
    adata,
    cluster_key=CLUSTER_KEY,
    plot_energy='total',
    order=CELL_TYPE_ORDER,
    colors=[colors[k] for k in CELL_TYPE_ORDER]
)
plt.show()


 ## 2.3 Energy on UMAP

In [ ]:
# Scatter plots of each energy component overlaid on UMAP embedding
sch.pl.plot_energy_scatters(
    adata,
    cluster_key=CLUSTER_KEY,
    basis='umap',
    show_legend=True
)
plt.show()


In [ ]:
# Interaction energy only
sch.pl.plot_energy_scatters(
    adata,
    cluster_key=CLUSTER_KEY,
    plot_energy='interaction'
)
plt.show()


 ## 2.4 Energy–Gene Correlations



 Identifies which genes drive energy differences across cell types.

In [ ]:
sch.tl.energy_gene_correlation(
    adata,
    spliced_key=SPLICED_KEY,
    cluster_key=CLUSTER_KEY
)

# Tabulate top correlated genes
df_correlations = sch.tl.get_correlation_table(
    adata,
    cluster_key=CLUSTER_KEY,
    energy_type='total',
    n_top_genes=100,
    order=CELL_TYPE_ORDER
)
print(df_correlations.head(10))


In [ ]:
# Pairwise scatter plots (all combinations of cell-type pairs)
sch.pl.plot_correlations_grid(
    adata,
    cluster_key=CLUSTER_KEY,
    energy='total',
    order=CELL_TYPE_ORDER,
    colors=colors,
    x_low=-0.4, x_high=0.4,
    y_low=-0.4, y_high=0.4
)
plt.show()


In [ ]:
# Highlight specific cell-type pairs
fig, ax = plt.subplots(1, 3, figsize=(18, 6), tight_layout=True)

sch.pl.plot_gene_correlation_scatter(
    adata, 'Meg', 'Ery',
    cluster_key=CLUSTER_KEY, energy='total',
    annotate=6, ax=ax[0],
    clus1_low=-0.4, clus1_high=0.4,
    clus2_low=-0.4, clus2_high=0.4
)
sch.pl.plot_gene_correlation_scatter(
    adata, 'Neu', 'Bas',
    cluster_key=CLUSTER_KEY, energy='total',
    annotate=6, ax=ax[1]
)
sch.pl.plot_gene_correlation_scatter(
    adata, 'Neu', 'Mon',
    cluster_key=CLUSTER_KEY, energy='total',
    annotate=6, ax=ax[2]
)
plt.show()


 ## 2.5 Corner Gene Identification



 "Corner genes" have high-magnitude correlation with one cell type and opposing

 (or low) correlation with another — these are candidate lineage-specific

 regulatory genes.

In [ ]:
genes_mask  = sch._utils.io.get_genes_used(adata)
gene_names  = adata.var.index[genes_mask]

# Build per-cluster correlation arrays
correlation = {}
for cluster in CELL_TYPE_ORDER:
    col = f'correlation_total_{cluster}'
    if col in adata.var.columns:
        correlation[cluster] = adata.var[col].values[genes_mask]

# Thresholds for "corner" classification
clus1_low  = -0.4
clus1_high =  0.4
clus2_low  = -0.4
clus2_high =  0.4
nn = 5   # top-n genes per pair

corner_genes = np.array([])

for corr1, corr2 in itertools.combinations(CELL_TYPE_ORDER, 2):
    if corr1 not in correlation or corr2 not in correlation:
        continue

    c1 = correlation[corr1]
    c2 = correlation[corr2]

    mask_corner = np.logical_or(
        np.logical_and(c1 >= clus1_high, c2 <= clus2_low),
        np.logical_and(c1 <= clus1_low,  c2 >= clus2_high)
    )

    idxs = np.where(mask_corner)[0]
    top  = np.argsort(c1[idxs] ** 2 + c2[idxs] ** 2)[-nn:]
    corner_genes = np.concatenate((corner_genes, gene_names[idxs[top]]))

corner_genes = np.unique(corner_genes)
print(f"Found {len(corner_genes)} corner genes:")
print(corner_genes)


In [ ]:
# Visualise corner gene correlation table
df_corr_corners = pd.DataFrame.from_dict(correlation, orient='columns')
df_corr_corners.index = gene_names
df_corr_corners = df_corr_corners.loc[corner_genes]
print(df_corr_corners.round(3))
